## Whisper-large-v3 → Yorùbá LoRA fine-tune (A100 80 GB profile)

Adapted from the Unsloth Whisper template for the Yorùbá voice-query pipeline.
Tuned for a large GPU (≈80 GB VRAM, ≥64 GB RAM): big batch, bf16, fused optimizer, parallel data prep, full epoch run.

**What this does**
- Loads `unsloth/whisper-large-v3` (≈3 GB in bf16) in 16-bit LoRA mode.
- Fine-tunes on `Hidi-agili/yoruba_tts_dataset` (diacritized Yorùbá, `audio,text`).
- Runs a full epoch by default (`num_train_epochs=3`, `max_steps=None`).

**Profile assumptions**
- ~80 GB VRAM (A100 / H100 / similar). For T4 free-tier, drop `per_device_train_batch_size` to 1 and re-enable `max_steps=60` (see git history).
- LoRA targets all attention + MLP projections for better Yorùbá adaptation.
- Data prep uses `datasets.map(num_proc=…)` to saturate CPU cores.


### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer torchcodec "datasets>=3.4.1,<4.0.0"

In [ ]:
# --- HF auth (only needed if you'll push to Hub at the end) ---
# Preferred: store the token in Colab Secrets (left sidebar → 🔑 key icon)
#   Name:  HF_TOKEN
#   Value: a write-permission token from https://huggingface.co/settings/tokens
#   Toggle "Notebook access" ON for this notebook.
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    # Not on Colab, or secret not set — fall back to env / interactive
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN  # what huggingface_hub reads
    from huggingface_hub import login
    login(token = HF_TOKEN, add_to_git_credential = False)
    print("HF auth: ok")
else:
    print("HF auth: skipped (no HF_TOKEN secret). Training will still work; "
          "you just won't be able to push_to_hub at the end.")

In [ ]:
from unsloth import FastModel
from transformers import WhisperForConditionalGeneration
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/whisper-large-v3",
    dtype = None, # auto detect
    load_in_4bit = False, # set True for 4-bit if VRAM is tight
    auto_model = WhisperForConditionalGeneration,
    whisper_language = "Yoruba",
    whisper_task = "transcribe",
    # token = "YOUR_HF_TOKEN", # only needed for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 64,
    # Cover all attention + MLP projections so the adapter can actually move
    # Yorùbá phoneme/diacritic distributions, not just Q/V.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "out_proj",
        "fc1", "fc2",
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    # ~80 GB VRAM: turn checkpointing OFF for ~20–30% throughput win.
    # Drop back to "unsloth" if you OOM (you won't at batch ≤48 on Hidi-agili).
    use_gradient_checkpointing = False,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    task_type = None, # ** MUST set this for Whisper **
)

<a name="Data"></a>
### Data Prep

We use `Hidi-agili/yoruba_tts_dataset` — the project's canonical Yorùbá speech set with diacritized transcripts (columns: `audio`, `text`). Audio is resampled to 16 kHz. A 6% test split is held out for eval during training.

To swap in another dataset, just change the `load_dataset` call below. Required columns: `audio` (HF Audio feature) and `text` (string transcript).

In [ ]:
import os
import numpy as np

# Yorùbá language token + transcribe task
model.generation_config.language = "<|yo|>"
model.generation_config.task = "transcribe"
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

feature_extractor = tokenizer.feature_extractor
text_tokenizer    = tokenizer.tokenizer

def formatting_prompts_func(example):
    audio = example["audio"]
    features = feature_extractor(audio["array"], sampling_rate = audio["sampling_rate"])
    tokenized_text = text_tokenizer(example["text"])
    return {
        "input_features": features.input_features[0],
        "labels": tokenized_text.input_ids,
    }

from datasets import load_dataset, Audio

# Project's canonical Yorùbá set (diacritized text, audio,text columns).
dataset = load_dataset("Hidi-agili/yoruba_tts_dataset", split = "train")
dataset = dataset.cast_column("audio", Audio(sampling_rate = 16000))
dataset = dataset.train_test_split(test_size = 0.06, seed = 3407)

# Saturate CPU for feature extraction — big speedup vs the per-example list comp.
NUM_PROC = max(1, (os.cpu_count() or 4) - 2)
print(f"Data prep: using num_proc={NUM_PROC}")

train_dataset = dataset["train"].map(
    formatting_prompts_func,
    remove_columns = dataset["train"].column_names,
    num_proc = NUM_PROC,
    desc = "Train split",
)
test_dataset = dataset["test"].map(
    formatting_prompts_func,
    remove_columns = dataset["test"].column_names,
    num_proc = NUM_PROC,
    desc = "Test split",
)
print(f"train={len(train_dataset)}  test={len(test_dataset)}")

In [ ]:
# @title Create compute_metrics and datacollator
import evaluate
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import pdb

metric = evaluate.load("wer")
def compute_metrics(pred):

    pred_logits = pred.predictions[0]
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id


    pred_ids = np.argmax(pred_logits, axis = -1)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens = True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens = True)

    wer = 100 * metric.compute(predictions = pred_str, references = label_str)

    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors = "pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors = "pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

<a name="Train"></a>
### Train the model
Full-epoch run tuned for ~80 GB VRAM: large per-device batch, bf16, fused AdamW, more dataloader workers. To revert to the T4 smoke run, set `per_device_train_batch_size=1`, `gradient_accumulation_steps=4`, `max_steps=60`, `num_train_epochs` unset.

In [ ]:
import os
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from unsloth import is_bf16_supported

DL_WORKERS = max(2, (os.cpu_count() or 4) // 2)

trainer = Seq2SeqTrainer(
    model = model,
    train_dataset = train_dataset,
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor = tokenizer),
    eval_dataset = test_dataset,
    tokenizer = tokenizer.feature_extractor,
    compute_metrics = compute_metrics,
    args = Seq2SeqTrainingArguments(
        # ---- throughput (A100 80GB, no grad checkpointing) ----
        per_device_train_batch_size = 48,
        per_device_eval_batch_size  = 48,
        gradient_accumulation_steps = 1,
        dataloader_num_workers      = DL_WORKERS,
        dataloader_pin_memory       = True,
        dataloader_persistent_workers = True,

        # ---- schedule ----
        # Hidi-agili is small (~1.1k clips). At batch 48 that's ~22 steps/epoch.
        # 3 epochs ≈ 66 steps total — finishes in a few minutes on A100.
        num_train_epochs = 3,
        max_steps        = -1,
        warmup_ratio     = 0.05,
        lr_scheduler_type = "cosine",
        learning_rate    = 1e-4,
        weight_decay     = 0.001,

        # ---- precision / optimizer ----
        bf16 = is_bf16_supported(),
        fp16 = not is_bf16_supported(),
        tf32 = True,
        optim = "adamw_torch_fused",

        # ---- logging / eval / checkpoints (cadence matches the small step count) ----
        logging_steps      = 2,
        eval_strategy      = "epoch",
        save_strategy      = "epoch",
        save_total_limit   = 2,
        load_best_model_at_end = True,
        metric_for_best_model  = "wer",
        greater_is_better      = False,

        # ---- misc ----
        remove_unused_columns = False,
        label_names = ["labels"],
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
3.012 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,123 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 31,457,280 of 1,574,947,840 (2.00% trained)
Unsloth: Not an error, but WhisperForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Wer
5,1.916300,1.587457,19.969040
10,1.441900,1.481879,19.504644
15,1.221700,1.397027,19.969040
20,2.508200,1.317600,20.046440
25,1.962400,1.243164,19.349845
30,1.558600,1.176595,20.046440
35,1.126300,1.122559,20.510836
40,1.012300,1.080295,21.439628
45,1.360200,1.044217,22.368421
50,1.133800,1.012587,23.297214


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

422.5309 seconds used for training.
7.04 minutes used for training.
Peak reserved memory = 10.348 GB.
Peak reserved memory for training = 7.336 GB.
Peak reserved memory % of max memory = 70.199 %.
Peak reserved memory for training % of max memory = 49.766 %.


<a name="Inference"></a>
### Inference
Sanity-check the fine-tuned model on a Yorùbá clip pulled from FLEURS (`yo_ng`, test split). FLEURS is the project's standard WER benchmark — see `scripts/eval_wer.py`.

In [ ]:
import soundfile as sf
from datasets import load_dataset, Audio
from IPython.display import Audio as AudioDisplay, display

fleurs = load_dataset("google/fleurs", "yo_ng", split = "test", streaming = True)
sample = next(iter(fleurs.cast_column("audio", Audio(sampling_rate = 16000))))
audio_file = "fleurs_yo_sample.wav"
sf.write(audio_file, sample["audio"]["array"], 16000)

print("Reference:", sample.get("transcription") or sample.get("raw_transcription"))
display(AudioDisplay(audio_file, rate = 16000))

In [ ]:
from transformers import pipeline
import torch

FastModel.for_inference(model)
model.eval()

whisper = pipeline(
    "automatic-speech-recognition",
    model = model,
    tokenizer = tokenizer.tokenizer,
    feature_extractor = tokenizer.feature_extractor,
    processor = tokenizer,
    return_language = True,
    torch_dtype = torch.float16,
)

# Force Yorùbá decoding for the demo
result = whisper(
    audio_file,
    generate_kwargs = {"language": "<|yo|>", "task": "transcribe"},
)
print("Hypothesis:", result["text"])

model.save_pretrained("whisper_yoruba_lora")  # Local saving
tokenizer.save_pretrained("whisper_yoruba_lora")
# Online saving (uses HF_TOKEN from the auth cell):
# model.push_to_hub("devalade/whisper-large-v3-yoruba-colab", token = HF_TOKEN)
# tokenizer.push_to_hub("devalade/whisper-large-v3-yoruba-colab", token = HF_TOKEN)

In [ ]:
model.save_pretrained("whisper_yoruba_lora")  # Local saving
tokenizer.save_pretrained("whisper_yoruba_lora")
# model.push_to_hub("devalade/whisper-large-v3-yoruba-colab", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("devalade/whisper-large-v3-yoruba-colab", token = "YOUR_HF_TOKEN")

# Merge to 16bit (full Whisper checkpoint — what `M1_HF_MODEL` in config.py expects)
if False: model.save_pretrained_merged("whisper_yoruba_16bit", tokenizer, save_method = None)
if False: model.push_to_hub_merged("devalade/whisper-large-v3-yoruba-colab", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)

# Merge to 4bit
if False: model.save_pretrained_merged("whisper_yoruba_4bit", tokenizer, save_method = "merged_4bit")
if False: model.push_to_hub_merged("devalade/whisper-large-v3-yoruba-colab-4bit", tokenizer, save_method = "merged_4bit", token = HF_TOKEN)

# Just LoRA adapters
if False:
    model.save_pretrained("whisper_yoruba_lora")
    tokenizer.save_pretrained("whisper_yoruba_lora")
if False:
    model.push_to_hub("devalade/whisper-yoruba-lora", token = HF_TOKEN)
    tokenizer.push_to_hub("devalade/whisper-yoruba-lora", token = HF_TOKEN)

In [ ]:
# Merge to 16bit (full Whisper checkpoint — what `M1_HF_MODEL` in config.py expects)
if False: model.save_pretrained_merged("whisper_yoruba_16bit", tokenizer, save_method = None)
if False: model.push_to_hub_merged("devalade/whisper-large-v3-yoruba-colab", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("whisper_yoruba_4bit", tokenizer, save_method = "merged_4bit")
if False: model.push_to_hub_merged("devalade/whisper-large-v3-yoruba-colab-4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("whisper_yoruba_lora")
    tokenizer.save_pretrained("whisper_yoruba_lora")
if False:
    model.push_to_hub("devalade/whisper-yoruba-lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("devalade/whisper-yoruba-lora", token = "YOUR_HF_TOKEN")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).